In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
# SOURCE AND TARGET PATHS

results_source_path = f"{s3_root_path}/Silver/Aaisha/results"
sprints_source_path = f"{s3_root_path}/Silver/Aaisha/sprints"

target_path = f"{s3_root_path}/Gold/Aaisha/fact_session_results"

print("Results Source :", results_source_path)
print("Sprints Source :", sprints_source_path)
print("Target Path    :", target_path)

In [0]:
results_df = (
    spark.read
        .format("delta")
        .load(results_source_path)
        .withColumn("session_type", F.lit("RACE"))
        .withColumnRenamed("position", "final_position")
        .drop(
            "raceName",
            "date",
            "ingestion_timestamp",
            "source_file"
        )
)

In [0]:
sprints_df = (
    spark.read
        .format("delta")
        .load(sprints_source_path)
        .withColumn("session_type", F.lit("SPRINT"))
        .withColumnRenamed("position", "final_position")
        .drop(
            "raceName",
            "date",
            "ingestion_timestamp",
            "source_file"
        )
)

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

display(results_sprints_df)

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

display(fact_session_results_df)

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(target_path)
)

In [0]:
gold_results_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_results_df)